In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    !pip install -q synth-cryo-em synth-core mrcfile gemmi nglview ipywidgets==7.7.1 matplotlib
    from google.colab import output

    output.enable_custom_widget_manager()
else:
    sys.path.append("../")

import matplotlib.pyplot as plt
import numpy as np

> **Note for Google Colab Users**: 
> When you run the setup cell above, Colab may display a blue bar or a popup asking to **'Enable third-party Jupyter widgets'**. 
> Please click **'Enable'** to allow the 3D visualization (nglview) to render correctly.

# Interactive Cryo-EM Simulation: Physics & Visualization

This tutorial demonstrates how to use `synth-cryo-em` to generate synthetic maps and visualize them interactively. 

We will explore:
1. **Map-to-Model Fit**: Overlaying the atomic model with its synthetic density.
2. **The Physics of Resolution**: How resolution, noise, and CTF effects change the visual features of a map.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from synth_core import add_gaussian_noise

from synth_cryo_em.core import apply_ctf, generate_density_map

## 1. Prepare the Data
We'll use a small standard PDB for this demonstration.

In [ ]:
# Use sample.pdb which contains actual atomic coordinates
if "google.colab" in sys.modules:
    pdb_path = "/content/synth-cryo-em/tests/sample.pdb"
else:
    pdb_path = "tests/sample.pdb"

if not os.path.exists(pdb_path):
    # Fallback for local run from different directory
    if os.path.exists("../tests/sample.pdb"):
        pdb_path = "../tests/sample.pdb"

print(f"Using model: {os.path.abspath(pdb_path)}")
if not os.path.exists(pdb_path):
    raise FileNotFoundError(f"Critical Error: PDB file not found at {pdb_path}")

## 2. Interactive Simulation Dashboard
Use the sliders below to adjust the physical parameters. The map will be re-generated and visualized automatically.

In [ ]:
def update_visuals(resolution, snr, defocus, apply_physics):
    global pdb_path
    if not os.path.exists(pdb_path):
        print(f"[DEBUG] ERROR: {pdb_path} does not exist!")
        return

    # 1. Generate core density
    grid, origin = generate_density_map(pdb_path, resolution=resolution)
    data = np.array(grid, copy=True)

    # 2. Apply Physics
    if apply_physics:
        uc = grid.unit_cell
        vox_size = (uc.a / grid.nu, uc.b / grid.nv, uc.c / grid.nw)
        data = apply_ctf(data, vox_size, defoc=defocus)

    # 3. Add Noise
    if snr < 50:  # Use 50 as 'clean' threshold
        data = add_gaussian_noise(data, snr=snr)

    # Visualization: Central 2D Slice
    # We extract the middle slice along the Z axis
    z_mid = data.shape[2] // 2
    slice_2d = data[:, :, z_mid]

    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(slice_2d, cmap="gray", origin="lower")
    ax.set_title(
        f"Central Z-Slice | Res={resolution}Å, SNR={snr}, Defocus={defocus}µm\nCTF={'ON' if apply_physics else 'OFF'}"
    )
    plt.colorbar(im, ax=ax, shrink=0.8, label="Density")
    plt.axis("off")
    plt.show()


# Widgets
res_slider = widgets.FloatSlider(
    value=4.0, min=2.0, max=10.0, step=0.5, description="Resolution (A)"
)
snr_slider = widgets.FloatSlider(value=50.0, min=1.0, max=50.0, step=1.0, description="SNR")
defocus_slider = widgets.FloatSlider(
    value=1.0, min=0.5, max=5.0, step=0.1, description="Defocus (um)"
)
physics_toggle = widgets.Checkbox(value=False, description="Apply CTF")
ui = widgets.VBox([res_slider, snr_slider, defocus_slider, physics_toggle])
out = widgets.interactive_output(
    update_visuals,
    {
        "resolution": res_slider,
        "snr": snr_slider,
        "defocus": defocus_slider,
        "apply_physics": physics_toggle,
    },
)
display(ui, out)